# AI Lab - Practical Question No. 14
## Uniform Cost Search (UCS) Implementation on Weighted Directed Graph

**Syllabus Area**: Uninformed Search Techniques  
**Difficulty**: 🟡 Intermediate  
**Core AI Concepts**: Uniform Cost Search (UCS), Priority Queue, Optimal Cost Search, Dijkstra's Algorithm Variant  
**Python Concepts Tested**: `heapq` module, min-heap invariant, tuple sorting, lazy deletion  
**Complexity**: Time: $O(E \log V)$, Space: $O(V)$  
**Sample Input**: `A->B:1, B->C:2, A->C:4. Start:A, Goal:C`  
**Sample Output**: `Path: A->B->C, Total Cost: 3`

---

### Problem Statement
Write a Python program to implement **Uniform Cost Search (UCS)** on a weighted directed graph to find the minimum cost path between a source and a destination. The algorithm must use a priority queue to always expand the lowest-cost node.

### Key Requirements
1. **Weighted Graph Representation**: Graph represented with explicit positive edge weights.
2. **Frontier Priority Queue**: Manage the frontier using Python's built-in `heapq` module.
3. **Cumulative Cost Tracking $g(n)$**: Compute and update path costs from start to current node.
4. **Optimal Path & Cost**: Return the reconstructed minimum-cost path and total integer/float cost.
5. **Goal Test Placement**: Test for goal upon **expansion** (popping from queue), not generation.
6. **Optional Challenge**: Detect negative edge weights and handle/trace negative cycles.

### 1. Theoretical Foundations & Algorithmic Principles

#### Why Uniform Cost Search?
- **Uninformed / Blind Search**: UCS requires no domain-specific heuristic $h(n)$. It expands nodes strictly in non-decreasing order of path cost $g(n)$ from the start node.
- **Dijkstra Equivalence**: UCS is essentially Dijkstra's algorithm formulated as a goal-directed search on graphs or tree spaces. While Dijkstra typically finds shortest paths to all vertices, UCS terminates as soon as the specific goal node is selected for expansion.
- **Tuple Sorting in Python**: `heapq` compares tuples element by element: `(cost, tie_breaker, node_id, path)`. By ordering by cumulative cost first and a tie-breaker second, the minimum-cost node is always popped in $O(\log N)$ time without comparison errors.
- **Expansion vs. Generation**: If goal testing was done upon generating an edge, a direct expensive path (e.g., $A \to C: 4$) would terminate prematurely, missing the optimal indirect path ($A \to B: 1, B \to C: 2$, cost 3). Expanding first guarantees optimality.

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is available in module path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.uniform_cost_search import (
    DEFAULT_SAMPLE_GRAPH,
    DIAMOND_BENCHMARK_GRAPH,
    ROMANIA_MAP_SUBGRAPH,
    CYCLIC_WEIGHTED_GRAPH,
    NEGATIVE_EDGE_DAG,
    NEGATIVE_CYCLE_GRAPH,
    uniform_cost_search,
    UniformCostSearch,
    detect_negative_cycle,
    parse_weighted_graph,
    visualize_ucs,
    get_ucs_viva_reference,
)

print("✓ Successfully imported UCS algorithms and benchmark datasets.")

### 2. Execution on Sample Input Graph (Question 14)

**Sample Graph Input**: `A->B:1, B->C:2, A->C:4`  
**Start Node**: `A` | **Goal Node**: `C`  
**Expected Output**: `Path: A->B->C, Total Cost: 3`

In [ ]:
sample_input = "A->B:1, B->C:2, A->C:4"
start_node = "A"
goal_node = "C"

result_sample = uniform_cost_search(sample_input, start=start_node, goal=goal_node)

print("=" * 65)
print(f"  Primary Output: {result_sample.format_path()}")
print("=" * 65)
print(f"Optimal Path:       {' -> '.join(result_sample.path)}")
print(f"Total Cost g(goal): {result_sample.total_cost}")
print(f"Nodes Expanded:     {', '.join(result_sample.nodes_expanded)}")
print(f"Execution Time:     {result_sample.execution_time_ms:.4f} ms\n")

print("Step-by-Step Priority Queue Trace Table:")
print(result_sample.format_step_table())

### 3. Cheaper Multi-Hop vs. Direct Expensive Path (Diamond Graph)

Demonstrates that Uniform Cost Search does not greedily take the path with fewer edges, but instead guarantees the lowest total path cost.
- Path 1 (2 hops): $S \to B \to G$, cost: $5 + 2 = 7$
- Path 2 (3 hops): $S \to A \to C \to G$, cost: $1 + 2 + 1 = 4$  ← **Optimal**

In [ ]:
result_diamond = uniform_cost_search(DIAMOND_BENCHMARK_GRAPH, start="S", goal="G")

print(f"Diamond Graph Result: {result_diamond.format_path()}")
print(f"Nodes Expanded:       {result_diamond.nodes_expanded}")
print("\nStep Table:")
print(result_diamond.format_step_table())

### 4. Classical AIMA Benchmark: Romania Driving Map

Computes the optimal driving route from **Arad** to **Bucharest** across the classic Russell & Norvig Romania roadmap.
Optimal Route: `Arad -> Sibiu (140) -> Rimnicu Vilcea (80) -> Pitesti (97) -> Bucharest (101)` = **418 km**.

In [ ]:
result_romania = uniform_cost_search(ROMANIA_MAP_SUBGRAPH, start="Arad", goal="Bucharest")

print(f"Romania Navigation Result: {result_romania.format_path()}")
print(f"Total Driving Distance:    {result_romania.total_cost} km")
print(f"Cities Expanded:           {', '.join(result_romania.nodes_expanded)}")

### 5. Optional Challenge: Negative Edge Weights & Negative Cycle Detection

UCS assumes non-negative edge costs ($c(u, v) \ge 0$). When a negative cycle is reachable, cumulative cost can decrease infinitely ($-\infty$).
Here we employ a Bellman-Ford cycle detection handler that traces and isolates the negative cycle.

In [ ]:
print("--- Part A: DAG with Negative Edge (No Cycle) ---")
res_neg_dag = uniform_cost_search(NEGATIVE_EDGE_DAG, start="S", goal="G")
print(f"Result:  {res_neg_dag.format_path()}")
print(f"Notice:  {res_neg_dag.warning_message}")

print("\n--- Part B: Reachable Negative Cycle (B -> C -> D -> B) ---")
has_cycle, cycle_path, _ = detect_negative_cycle(NEGATIVE_CYCLE_GRAPH, start_node="A")
print(f"Negative Cycle Detected: {has_cycle}")
print(f"Cycle Vertices:          {' -> '.join(cycle_path or [])}")

res_cycle = uniform_cost_search(NEGATIVE_CYCLE_GRAPH, start="A", goal="Goal")
print(f"UCS Output with Cycle:   {res_cycle.format_path()}")
print(f"Diagnostics:             {res_cycle.warning_message}")

### 6. Graphical Visualization of Search Graph & Optimal Path

Generates an annotated NetworkX / Matplotlib visual representation displaying node roles (Start, Goal, Path, Explored) and edge costs.

In [ ]:
output_image = repo_root / "reports" / "ucs_sample_path.png"
saved_path = visualize_ucs(result_sample, output_path=output_image, show_plot=False)
print(f"✓ Visual diagram successfully saved to: {saved_path}")

### 7. Viva Voce Theory & Examination Guide

Authoritative answers for standard university viva voce examination topics.

In [ ]:
viva_guide = get_ucs_viva_reference()

for i, item in enumerate(viva_guide["core_topics"], start=1):
    print(f"[{i}] {item['topic'].upper()}")
    print(f"Q: {item['question']}")
    print("A:")
    for line in item['answer'].split('\n'):
        print(f"   {line}")
    print("-" * 70)

matrix = viva_guide["comparison_matrix"]
print(f"\n{' | '.join(matrix['headers'])}")
print("=" * 115)
for row in matrix["rows"]:
    print(f"{row[0]:<18} | {row[1]:<32} | {row[2]:<10} | {row[3]:<12} | {row[4]:<22} | {row[5]}")